In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# Project 1 — Local PDF Q&A Tutor
## Chat with PDFs Using Local Embeddings and Ollama

**What you'll learn:**
- Load and parse PDF documents with PyPDF
- Split text into overlapping chunks for better retrieval
- Create embeddings with a local Ollama model
- Store vectors in ChromaDB
- Build a RetrievalQA chain that answers questions with citations

**Stack:** Ollama · LangChain · ChromaDB · PyPDF · Jupyter

**Prerequisites:** `ollama pull nomic-embed-text` and `ollama pull qwen3:8b`

```
PDF → Split into chunks → Embed each chunk → Store in Chroma
                                                       ↓
User question → Embed question → Find similar chunks → Feed to LLM → Answer
```

In [ ]:
# Cell 1 — Install dependencies
# !pip install -q langchain langchain-ollama langchain-community chromadb pypdf

## Step 1 — Configure LLM and Embeddings

In [ ]:
from langchain_ollama import ChatOllama, OllamaEmbeddings

llm = ChatOllama(model="qwen3:8b", temperature=0.3)
embeddings = OllamaEmbeddings(model="nomic-embed-text")

vec = embeddings.embed_query("test")
print(f"Embedding dim: {len(vec)} — model is working!")

## Step 2 — Create a Sample PDF (self-contained demo)

In [ ]:
from pathlib import Path
import textwrap

SAMPLE_DIR = Path("sample_data"); SAMPLE_DIR.mkdir(exist_ok=True)

sample_text = textwrap.dedent("""
    Machine Learning Fundamentals

    Supervised learning uses labeled data to train models that predict outcomes.
    Common algorithms include linear regression for continuous targets and
    logistic regression for classification. Decision trees split data based on
    feature thresholds, while random forests combine many trees to reduce overfitting.

    Deep Learning Overview

    Neural networks consist of layers of interconnected nodes. Convolutional
    Neural Networks (CNNs) excel at image recognition. Recurrent Neural Networks
    (RNNs) and LSTMs handle sequential data such as text and time series.

    Transformers and Attention

    The transformer architecture replaced recurrence with self-attention mechanisms.
    This enables parallel processing, leading to BERT for understanding and GPT for
    generation. Large Language Models are scaled-up transformers.

    Retrieval-Augmented Generation (RAG)

    RAG combines a retriever with a generator. This reduces hallucination by grounding
    the LLM in actual source material. Key components: document chunking, embedding
    models, vector databases, and prompt engineering.
""")
(SAMPLE_DIR / "ml_fundamentals.txt").write_text(sample_text, encoding="utf-8")
print("Sample document saved.")

## Step 3 — Load and Chunk the Document
`RecursiveCharacterTextSplitter` splits on paragraph → sentence → word boundaries.

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

loader = TextLoader("sample_data/ml_fundamentals.txt", encoding="utf-8")
docs = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " "])
chunks = splitter.split_documents(docs)

print(f"Original: {len(docs[0].page_content)} chars → {len(chunks)} chunks")
for i, c in enumerate(chunks):
    print(f"  Chunk {i}: {len(c.page_content)} chars — {c.page_content[:60]}...")

## Step 4 — Create the Vector Store with ChromaDB

In [ ]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks, embedding=embeddings,
    persist_directory="sample_data/chroma_db",
    collection_name="pdf_qa_tutor",
)
results = vectorstore.similarity_search("What is RAG?", k=2)
print("Top 2 results for 'What is RAG?':")
for i, r in enumerate(results):
    print(f"  [{i+1}] {r.page_content[:100]}...")

## Step 5 — Build the QA Chain with Citations

In [ ]:
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

prompt_template = PromptTemplate(
    input_variables=["context", "question"],
    template="""Use the following context to answer the question. If you cannot
find the answer, say "I don't have enough information."

Context:
{context}

Question: {question}

Answer (cite relevant details from the context):"""
)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff",
    retriever=vectorstore.as_retriever(search_kwargs={"k": 3}),
    return_source_documents=True,
    chain_type_kwargs={"prompt": prompt_template},
)
print("QA chain ready!")

## Step 6 — Ask Questions

In [ ]:
questions = [
    "What is the difference between CNNs and RNNs?",
    "How does RAG reduce hallucination?",
    "What did the transformer architecture replace?",
]
for q in questions:
    print(f"\n{'='*60}\nQ: {q}")
    result = qa_chain.invoke({"query": q})
    print(f"A: {result['result']}")
    print(f"Sources: {len(result['source_documents'])} chunks used")

## Step 7 — Interactive Q&A Helper

In [ ]:
def ask(question: str) -> str:
    result = qa_chain.invoke({"query": question})
    answer = result["result"]
    sources = result["source_documents"]
    output = f"Answer: {answer}\n\nSources ({len(sources)}):\n"
    for i, src in enumerate(sources):
        output += f"  [{i+1}] {src.page_content[:80]}...\n"
    return output

print(ask("What are the key components of RAG?"))

## What You Learned
- **PDF loading** → text extraction
- **Chunking** → overlapping splits for context retention
- **Local embeddings** → Ollama nomic-embed-text
- **Vector store** → ChromaDB for similarity search
- **RetrievalQA** → end-to-end Q&A with citations